# Data quality checks

Runs after the Gold build in `Refresh_Orchestration` and before the semantic model refresh. Every check compares one
layer with another or with a plausibility rule, so a silent loss, duplication or corruption of data is caught on the
day it happens.

| Category | What is checked |
|---|---|
| Freshness | Last successful Strava sync, newest workout reached Gold, fitness curve up to today |
| Completeness | Bronze → Silver → Gold workout counts and distance reconcile; one calendar row per day; backfill progress |
| Integrity | Unique keys; every workout has a date, activity type and source in the dimensions; Period slicer bridge |
| Validity | Plausible speed, heart rate, power and duration; no future workouts; unmapped activity types; rejected records |
| Security | No plaintext token table in any lakehouse (credentials live in Key Vault) |
| Metrics | Training-load coverage and fitness (CTL) range |

Output (lakehouse `Strava_Gold`): `DataQualityCheck` (latest run), `DataQualityHistory` (all runs) and `DataStatus`
(one row behind the report's data-status footer). The notebook exits with
`{"status": "passed" | "warnings" | "failed"}`; the pipeline fails the run when an error-severity check fails, after
the report has been refreshed, so the failure is visible in both places.

In [ ]:
stale_sync_warning_hours = 30
stale_sync_error_hours = 72

In [ ]:
from datetime import datetime, timezone, timedelta
import json
import uuid

from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.parquet.vorder.default", "true")

WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
RUN_ID = str(uuid.uuid4())
CHECKED_AT = datetime.now(timezone.utc).replace(microsecond=0)
TODAY = CHECKED_AT.date()


def lakehouse_id(name):
    """Resolve a lakehouse in this workspace by name, so the notebook works in any workspace."""
    try:
        item = notebookutils.lakehouse.get(name)
        return item["id"] if isinstance(item, dict) else item.id
    except Exception:
        import sempy.fabric as fabric
        return fabric.resolve_item_id(name, type="Lakehouse")


ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
BRONZE = f"{ONELAKE}/{lakehouse_id('Bronze')}/Tables"
SILVER = f"{ONELAKE}/{lakehouse_id('Strava_Silver')}/Tables"


def exists(path):
    try:
        return notebookutils.fs.exists(f"{path}/_delta_log")
    except Exception:
        return False


def read(path):
    return spark.read.format("delta").load(path) if exists(path) else None


def gold(name):
    return read(f"Tables/{name}")


results = []


def check(category, name, description, severity, passed, observed, expected, detail=""):
    """Record one check. severity: error (fails the run) | warning | info. passed=None means not applicable."""
    status = "skipped" if passed is None else ("pass" if passed else ("fail" if severity == "error" else ("warn" if severity == "warning" else "info")))
    results.append((RUN_ID, CHECKED_AT, category, name, description, severity, status,
                    None if observed is None else str(observed), None if expected is None else str(expected), detail))
    print(f"[{status.upper():7}] {category:12} {name}: observed={observed} expected={expected} {detail}")


def safe(category, name, description, severity, fn):
    """Run a check function; a crash is itself a failed check instead of an aborted notebook."""
    try:
        fn()
    except Exception as exc:  # noqa: BLE001 - every unexpected problem must surface as a result row
        check(category, name, description, severity, False, "error", "check runs", f"{type(exc).__name__}: {str(exc)[:300]}")

In [ ]:
# Inputs
run_log = read(f"{BRONZE}/meta/ingestion_run_log")
bronze_summaries = read(f"{BRONZE}/strava/athlete_activities")
silver_workout = read(f"{SILVER}/Workout")
silver_issues = read(f"{SILVER}/DataQualityIssue")
fact_workout = gold("FactWorkout")
daily = gold("FactDailyActivity")
fitness = gold("FactFitnessDaily")
dim_date = gold("DimDate")
dim_type = gold("DimActivityType")
dim_source = gold("DimSourceSystem")
dim_period = gold("DimPeriod")

last_sync = None
last_run = None
if run_log is not None:
    rows = (run_log.where((F.col("source_system") == "strava") & F.col("status").isin("success", "partial"))
            .orderBy(F.desc("finished_at")).limit(1).collect())
    if rows:
        last_run = rows[0]
        last_sync = last_run["finished_at"].replace(tzinfo=timezone.utc) if last_run["finished_at"].tzinfo is None else last_run["finished_at"]
last_details = json.loads(last_run["details_json"]) if last_run and last_run["details_json"] else {}


def rejected_workout_ids():
    """Strava activities Silver rejected as unusable (logged in DataQualityIssue) - they never reach Gold by design."""
    if silver_issues is None:
        return spark.createDataFrame([], "source_record_id string")
    return (silver_issues.where("source_system = 'strava' AND entity_name = 'workout' AND severity = 'error'")
            .select("source_record_id").distinct())

In [ ]:
# Freshness
def freshness_sync():
    if last_sync is None:
        check("Freshness", "Strava sync", "A Strava ingestion run finished successfully", "error", False, "never", "a successful run")
        return
    age_h = round((CHECKED_AT - last_sync).total_seconds() / 3600, 1)
    severity = "error" if age_h > stale_sync_error_hours else "warning"
    check("Freshness", "Strava sync", "Hours since the last successful Strava sync", severity,
          age_h <= stale_sync_warning_hours, f"{age_h} h", f"<= {stale_sync_warning_hours} h")


def freshness_latest_workout():
    if bronze_summaries is None:
        check("Freshness", "Newest workout in report", "The newest Strava workout reached the Gold layer", "error", None,
              "no activities yet", "-")
        return
    bronze_max = (bronze_summaries.join(rejected_workout_ids(), "source_record_id", "left_anti")
                  .select(F.max(F.to_date(F.get_json_object("raw_payload", "$.start_date_local")))).collect()[0][0])
    gold_max = fact_workout.select(F.max("activity_date")).collect()[0][0]
    check("Freshness", "Newest workout in report", "The newest Strava workout reached the Gold layer", "error",
          bronze_max is None or (gold_max is not None and gold_max >= bronze_max), gold_max, f">= {bronze_max}")


def freshness_fitness():
    last = fitness.select(F.max("date")).collect()[0][0]
    check("Freshness", "Fitness curve current", "Fitness / fatigue / form are calculated up to today", "warning",
          last is not None and last >= TODAY - timedelta(days=1), last, f">= {TODAY - timedelta(days=1)}")


safe("Freshness", "Strava sync", "", "error", freshness_sync)
safe("Freshness", "Newest workout in report", "", "error", freshness_latest_workout)
safe("Freshness", "Fitness curve current", "", "warning", freshness_fitness)

In [ ]:
# Completeness: Bronze -> Silver -> Gold reconciliation
def bronze_to_silver():
    if bronze_summaries is None:
        check("Completeness", "Bronze to Silver workouts", "Every landed Strava activity is in Silver or logged as rejected",
              "error", None, "no activities yet", "-")
        return
    bronze_ids = bronze_summaries.select("source_record_id").distinct()
    silver_ids = silver_workout.where("source_system = 'strava'").select(F.col("source_workout_id").alias("source_record_id"))
    rejected = rejected_workout_ids()
    missing = bronze_ids.join(silver_ids.unionByName(rejected).distinct(), "source_record_id", "left_anti").count()
    check("Completeness", "Bronze to Silver workouts", "Every landed Strava activity is in Silver or logged as rejected",
          "error", missing == 0, f"{missing} missing", "0 missing", f"{bronze_ids.count()} activities landed")


def silver_to_gold():
    primary = silver_workout.where("is_primary_record")
    s = primary.agg(F.count("*").alias("n"), F.sum("distance_m").alias("m")).collect()[0]
    g = fact_workout.agg(F.count("*").alias("n"), F.sum("distance").alias("m")).collect()[0]
    check("Completeness", "Silver to Gold workouts", "Gold has exactly one row per primary Silver workout", "error",
          s["n"] == g["n"], g["n"], s["n"])
    sm, gm = float(s["m"] or 0), float(g["m"] or 0)
    check("Completeness", "Silver to Gold distance", "Total distance is identical in Silver and Gold", "error",
          abs(sm - gm) <= max(1.0, sm * 0.001), f"{gm / 1000:,.1f} km", f"{sm / 1000:,.1f} km")


def calendar_complete():
    agg = daily.agg(F.min("date_key").alias("d0"), F.max("date_key").alias("d1"), F.count("*").alias("n"),
                    F.countDistinct("date_key").alias("u")).collect()[0]
    expected = (agg["d1"] - agg["d0"]).days + 1 if agg["d0"] else 0
    check("Completeness", "Daily calendar", "One activity row per calendar day, rest days included, up to today", "error",
          agg["n"] == expected == agg["u"] and agg["d1"] is not None and agg["d1"] >= TODAY - timedelta(days=1),
          f"{agg['n']} days to {agg['d1']}", f"{expected} days to {TODAY}")


def backfill():
    remaining = last_details.get("backfill_remaining") or {}
    details, streams = remaining.get("activity_detail"), remaining.get("activity_streams")
    check("Completeness", "History backfill", "Detailed data (laps, efforts, heart-rate / power streams) loaded for all activities",
          "info", None if details is None else (details == 0 and streams == 0),
          f"{details} details, {streams} streams to go", "0",
          "Strava allows ~1,000 requests a day, so a long history fills in over several daily runs.")


safe("Completeness", "Bronze to Silver workouts", "", "error", bronze_to_silver)
safe("Completeness", "Silver to Gold workouts", "", "error", silver_to_gold)
safe("Completeness", "Daily calendar", "", "error", calendar_complete)
safe("Completeness", "History backfill", "", "info", backfill)

In [ ]:
# Integrity
def unique(df, key, category, name, description):
    n, u = df.agg(F.count("*"), F.countDistinct(key)).collect()[0]
    check(category, name, description, "error", n == u, f"{n - u} duplicates", "0 duplicates")


def orphans(child, child_key, parent, parent_key, name, description):
    missing = (child.where(F.col(child_key).isNotNull()).select(F.col(child_key).alias("k")).distinct()
               .join(parent.select(F.col(parent_key).alias("k")), "k", "left_anti").count())
    check("Integrity", name, description, "error", missing == 0, f"{missing} unmatched", "0 unmatched")


safe("Integrity", "Unique workouts", "", "error",
     lambda: unique(fact_workout, "workout_key", "Integrity", "Unique workouts", "No workout appears twice in Gold"))
safe("Integrity", "Unique dates", "", "error",
     lambda: unique(dim_date, "date_key", "Integrity", "Unique dates", "The date dimension has one row per day"))
safe("Integrity", "Workout dates", "", "error",
     lambda: orphans(fact_workout, "date_key", dim_date, "date_key", "Workout dates", "Every workout date exists in the date dimension"))
safe("Integrity", "Workout activity types", "", "error",
     lambda: orphans(fact_workout, "activity_type_key", dim_type, "activity_type_key", "Workout activity types",
                     "Every workout's activity type exists in the activity type dimension"))
safe("Integrity", "Workout sources", "", "error",
     lambda: orphans(fact_workout, "source_system_key", dim_source, "source_system_key", "Workout sources",
                     "Every workout's source exists in the source dimension"))


def period_bridge():
    # DimPeriod ends on the day Gold_Longevity_Metrics ran; allow one day in case the run crossed midnight UTC.
    ends = dim_period.groupBy("period_label").agg(F.max("date_key").alias("end")).collect()
    default = dim_period.where("is_default").select("period_label").distinct().count()
    end_dates = {r["end"] for r in ends}
    current = len(end_dates) == 1 and min(end_dates) >= TODAY - timedelta(days=1)
    check("Integrity", "Period slicer", "Six report periods, one default, all ending on the latest day", "error",
          len(ends) == 6 and default == 1 and current, f"{len(ends)} periods ending {', '.join(map(str, sorted(end_dates)))}",
          f"6 periods ending {TODAY - timedelta(days=1)} or later")


safe("Integrity", "Period slicer", "", "error", period_bridge)

In [ ]:
# Validity (plausibility of recorded values)
SPEED_LIMITS = "CASE activity_family WHEN 'cycling' THEN 70 WHEN 'running' THEN 25 WHEN 'walking' THEN 12 WHEN 'swimming' THEN 8 ELSE 200 END"
validity = [
    ("Future workouts", "No workout is dated in the future", "warning", f"activity_date > date_add(current_date(), 1)"),
    ("Moving time recorded", "Every workout has a moving time (Strava's auto-pause can record 0)", "warning",
     "duration_minutes IS NULL OR duration_minutes <= 0"),
    ("Plausible speed", "Average speed is realistic for the sport", "warning", f"avg_speed_kmh > {SPEED_LIMITS}"),
    ("Plausible heart rate", "Average heart rate is between 30 and 230 bpm", "warning",
     "average_heartrate IS NOT NULL AND (average_heartrate < 30 OR average_heartrate > 230)"),
    ("Plausible power", "Average power is below 1,500 W", "warning", "average_watts > 1500"),
]
for name, description, severity, condition in validity:
    def _validity(name=name, description=description, severity=severity, condition=condition):
        bad = fact_workout.where(condition)
        n = bad.count()
        examples = bad.orderBy(F.desc("activity_date")).select("activity_date", "name").limit(3).collect()
        detail = "; ".join(f"{r['activity_date']} {r['name']}" for r in examples) + (" ..." if n > 3 else "")
        check("Validity", name, description, severity, n == 0, f"{n} workouts", "0",
              (detail + " - fix or delete in Strava") if n else "")
    safe("Validity", name, description, severity, _validity)


def unmapped_types():
    rows = (silver_workout.where("activity_type = 'other' AND source_activity_type IS NOT NULL")
            .groupBy("source_activity_type").count().orderBy(F.desc("count")).limit(5).collect())
    check("Validity", "Activity types mapped", "Every Strava sport type maps to a known activity type", "warning",
          not rows, f"{sum(r['count'] for r in rows)} workouts", "0", ", ".join(r["source_activity_type"] for r in rows))


def rejected_records():
    if silver_issues is None:
        check("Validity", "Rejected records", "Records excluded because they are unusable", "warning", None, None, "0")
        return
    rows = silver_issues.where("severity = 'error' AND entity_name = 'workout'").groupBy("rule_code").count().collect()
    check("Validity", "Rejected records", "Workouts excluded because they are unusable (e.g. no start time or duration)",
          "warning", not rows, f"{sum(r['count'] for r in rows)} workouts", "0", ", ".join(f"{r['rule_code']}: {r['count']}" for r in rows))


def api_errors():
    errors = (last_details.get("counts") or {}).get("errors")
    failed = (silver_issues.where("severity = 'error' AND entity_name <> 'workout'").groupBy("entity_name", "rule_code").count().collect()
              if silver_issues is not None else [])
    check("Validity", "Strava API errors", "The last sync had no failed API calls", "warning",
          None if errors is None else errors == 0, errors, "0",
          ("Still failing: " + ", ".join(f"{r['entity_name']} {r['rule_code']}: {r['count']}" for r in failed)
           + ". HTTP 401/403 usually means a missing permission - reconnect with python tools/strava_auth.py") if failed else "")


safe("Validity", "Activity types mapped", "", "warning", unmapped_types)
safe("Validity", "Rejected records", "", "warning", rejected_records)
safe("Validity", "Strava API errors", "", "warning", api_errors)

In [ ]:
# Security and metric sanity
def no_plaintext_tokens():
    found = [p for p in ["strava/auth_tokens", "meta/auth_tokens", "dbo/auth_tokens", "auth_tokens"] if exists(f"{BRONZE}/{p}")]
    check("Security", "No stored tokens", "No access or refresh token table exists in a lakehouse (secrets live in Key Vault)",
          "error", not found, ", ".join(found) or "none", "none")


def load_coverage():
    recent = fact_workout.where(F.col("activity_date") >= F.date_sub(F.current_date(), 90))
    n, with_load = recent.agg(F.count("*"), F.count("training_load_score")).collect()[0]
    pct = round(with_load / n * 100) if n else None
    check("Metrics", "Training load coverage", "Workouts of the last 90 days that have a training load (power or heart rate)",
          "warning", None if not n else pct >= 50, None if pct is None else f"{pct}%", ">= 50%",
          "Without power or heart-rate data a workout adds nothing to fitness / fatigue / form.")


def ctl_range():
    ctl = fitness.orderBy(F.desc("date")).select("fitness_ctl").limit(1).collect()
    value = ctl[0][0] if ctl else None
    check("Metrics", "Fitness range", "Current fitness (CTL) is within a realistic range", "warning",
          None if value is None else 0 <= value <= 250, value, "0-250")


safe("Security", "No stored tokens", "", "error", no_plaintext_tokens)
safe("Metrics", "Training load coverage", "", "warning", load_coverage)
safe("Metrics", "Fitness range", "", "warning", ctl_range)

In [ ]:
# Results
RESULT_DDL = ("run_id string, checked_at timestamp, category string, check_name string, description string, severity string, "
              "status string, observed string, expected string, detail string")
STATUS_ORDER = F.create_map(*[x for kv in {"fail": 1, "warn": 2, "info": 3, "skipped": 4, "pass": 5}.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
checks = (spark.createDataFrame(results, RESULT_DDL)
          .withColumn("status_sort", STATUS_ORDER[F.col("status")])
          .withColumn("status_label", F.when(F.col("status") == "pass", "Passed").when(F.col("status") == "fail", "Failed")
                      .when(F.col("status") == "warn", "Warning").when(F.col("status") == "info", "Info").otherwise("Not applicable")))
checks.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/DataQualityCheck")
checks.write.mode("append").format("delta").option("mergeSchema", "true").save("Tables/DataQualityHistory")

statuses = [r[6] for r in results]
errors, warnings, passed = statuses.count("fail"), statuses.count("warn"), statuses.count("pass")
evaluated = len([s for s in statuses if s in ("pass", "fail", "warn")])
overall = "failed" if errors else ("warnings" if warnings else "passed")
latest_workout = fact_workout.select(F.max("activity_date")).collect()[0][0] if fact_workout is not None else None
remaining = last_details.get("backfill_remaining") or {}
status_text = {"passed": "All data checks passed", "warnings": f"{warnings} data check warning{'s' if warnings != 1 else ''}",
               "failed": f"{errors} data check{'s' if errors != 1 else ''} failed"}[overall]
spark.createDataFrame([(
    RUN_ID, CHECKED_AT, last_sync, latest_workout, len(results), passed, warnings, errors, evaluated,
    {"passed": "Healthy", "warnings": "Check warnings", "failed": "Needs attention"}[overall], status_text,
    remaining.get("activity_detail"), remaining.get("activity_streams"),
)], "run_id string, checked_at timestamp, last_sync_utc timestamp, latest_workout_date date, checks_total int, "
    "checks_passed int, checks_warning int, checks_failed int, checks_evaluated int, overall_status string, "
    "status_text string, details_to_backfill long, streams_to_backfill long") \
    .write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/DataStatus")

summary = {"status": overall, "run_id": RUN_ID, "checks": len(results), "passed": passed, "warnings": warnings, "errors": errors,
           "failed_checks": [r[3] for r in results if r[6] == "fail"]}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))